# 01 — Thermostats and barostats

**Learning goals**

- set up a small periodic 2D Lennard-Jones fluid with the minimum-image convention,
- compare NVE dynamics, repeated velocity rescaling and a Langevin thermostat,
- see that a thermostat changes both the temperature statistics *and* the dynamics,
- watch a weak-coupling barostat respond to a change in target pressure.

Reduced LJ units: $\epsilon=\sigma=m=k_B=1$.

In [ ]:
# Setup: make the `src` package importable (locally or on Google Colab).
import sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/HyeonjeYang/interactive-metadynamics.git"
if "google.colab" in sys.modules:
    ROOT = Path("/content/interactive-metadynamics")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
else:
    ROOT = Path.cwd().resolve()
    if not (ROOT / "src").is_dir():
        ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import inspect
import numpy as np
import matplotlib.pyplot as plt

from src import lj
from src.thermal import (maxwell_boltzmann_velocities, remove_com_velocity,
                         rescale_to_temperature, degrees_of_freedom)
from src.plotting import set_style, save_figure, PALETTE, REFERENCE, MUTED
set_style()

## Parameters

$N = 36$ is used instead of 16 because the minimum-image convention needs a box length $L > 2r_c = 5\sigma$;
at liquid-like densities that requires roughly $N \gtrsim 25$ particles. The force loop is still trivially cheap.

In [ ]:
n_side = 6                 # N = n_side**2 particles on an initial square lattice
density = 0.5              # N / A
T_target = 1.0
dt = 0.005
n_steps = 8000             # 40 time units per thermostat run
sample_stride = 10
friction = 1.0             # Langevin friction coefficient zeta (damping rate zeta/m)

P_targets = (1.0, 2.0)     # barostat target pressure, switched halfway through the run
tau_P = 2.0                # barostat coupling time
compressibility = 1.0      # only compressibility / tau_P matters for Berendsen coupling
barostat_steps = 8000

seed = 2026
rng = np.random.default_rng(seed)

## 1. The model

Pair potential, truncated at $r_c=2.5\sigma$ and shifted so that $V(r_c)=0$ (no tail corrections):

$$V(r) = 4\epsilon\left[\left(\frac{\sigma}{r}\right)^{12} - \left(\frac{\sigma}{r}\right)^{6}\right] - V_\mathrm{LJ}(r_c), \qquad r < r_c.$$

Periodic boundaries: each separation vector is mapped to its nearest image, $\Delta\mathbf r \leftarrow \Delta\mathbf r - L\,\mathrm{round}(\Delta\mathbf r/L)$.
The instantaneous pressure in $d=2$ uses the virial $W=\sum_{i<j}\mathbf r_{ij}\cdot\mathbf F_{ij}$:

$$P = \frac{N k_B T_\mathrm{inst} + W/d}{A}.$$

In [ ]:
print(inspect.getsource(lj.minimum_image))
print(inspect.getsource(lj.pressure))

## 2. Initial state

Lattice positions, Maxwell–Boltzmann velocities, centre-of-mass removal and one initial rescaling (Notebook 00).

In [ ]:
N = n_side**2
L = np.sqrt(N / density)
dof = degrees_of_freedom(N, 2)
x_init = lj.square_lattice(n_side, L)
v_init = remove_com_velocity(maxwell_boltzmann_velocities(1.0, T_target, N, 2, rng), 1.0)
v_init, _ = rescale_to_temperature(v_init, 1.0, T_target, dof)
print(f"N = {N}, L = {L:.3f} (2 r_c = 5.0), f = {dof}, T_inst = {lj.internal_temperature(v_init):.3f}")

## 3. NVE: Velocity Verlet without a thermostat

In [ ]:
nve = lj.run_lj(x_init, v_init, L, n_steps, dt, thermostat="none", sample_stride=sample_stride)
t = nve["time"]

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
axes[0].plot(t, nve["kinetic"] / N, label="kinetic")
axes[0].plot(t, nve["potential"] / N, label="potential")
axes[0].plot(t, nve["total"] / N, color=REFERENCE, label="total")
axes[0].set_xlabel("time")
axes[0].set_ylabel("energy per particle")
axes[0].legend()
axes[1].scatter(*x_init.T, s=30, color=MUTED, label="initial lattice")
axes[1].scatter(*nve["positions"].T, s=30, color=PALETTE[0], label="final")
axes[1].set_aspect("equal")
axes[1].set_xlim(0, L); axes[1].set_ylim(0, L)
axes[1].set_title("periodic box")
axes[1].legend(loc="upper right", fontsize=8)
fig.tight_layout()
plt.show()

E = nve["total"]
print(f"total-energy std / N = {E.std() / N:.1e},  drift / N = {(E[-1] - E[0]) / N:.1e}")
print(f"<T_inst> over the second half = {nve['temperature'][len(t) // 2:].mean():.3f}  (started at {T_target})")

The total energy is conserved to about $10^{-4}$ per particle at this timestep (the small residual comes from
the finite $\Delta t$ and the force discontinuity at the cutoff). The temperature, however, is *not* controlled:
as the lattice melts, potential energy turns into kinetic energy and $T_\mathrm{inst}$ rises above the starting value.

## 4. Repeated exact velocity rescaling (pedagogical contrast)

After every step, $\mathbf v \leftarrow \lambda\mathbf v$ with $\lambda=\sqrt{T_\mathrm{target}/T_\mathrm{inst}}$.
This holds the kinetic temperature *exactly* constant. It does **not** generate the canonical ensemble: in a canonical
system the kinetic energy fluctuates.

In [ ]:
rescale = lj.run_lj(x_init, v_init, L, n_steps, dt, thermostat="rescale", T_target=T_target,
                    sample_stride=sample_stride)

## 5. Langevin thermostat (BAOAB splitting)

Underdamped Langevin dynamics $m\dot{\mathbf v} = \mathbf F - \zeta\mathbf v + \sqrt{2\zeta k_BT}\,\boldsymbol\eta(t)$,
split into kicks (B), drifts (A) and an exact Ornstein–Uhlenbeck velocity update (O):

$$\text{B: } \mathbf v \mathrel{+}= \tfrac{\Delta t}{2}\mathbf F/m \quad
\text{A: } \mathbf x \mathrel{+}= \tfrac{\Delta t}{2}\mathbf v \quad
\text{O: } \mathbf v \leftarrow c_1\mathbf v + \sqrt{(1-c_1^2)k_BT/m}\,\boldsymbol\xi, \;\; c_1=e^{-\zeta\Delta t/m}$$

applied as B-A-O-A-B. Friction and noise do not conserve total momentum, so the temperature is measured
in the centre-of-mass frame with $f = dN - d$.

In [ ]:
langevin = lj.run_lj(x_init, v_init, L, n_steps, dt, thermostat="langevin", T_target=T_target,
                     friction=friction, rng=rng, sample_stride=sample_stride)

## 6. Comparison: temperature control and fluctuations

In [ ]:
runs = {"NVE": nve, "velocity rescaling": rescale, "Langevin (BAOAB)": langevin}

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), gridspec_kw={"width_ratios": [2, 1]})
axes[0].axhline(T_target, color=REFERENCE, ls="--", lw=1, label="target", zorder=1)
for name, r in runs.items():
    axes[0].plot(r["time"], r["temperature"], lw=1.2, label=name)
axes[0].set_xlabel("time")
axes[0].set_ylabel(r"$T_\mathrm{inst}$")
axes[0].legend(ncol=2, fontsize=8)

half = slice(len(t) // 2, None)
for name, r in runs.items():
    if name != "velocity rescaling":
        axes[1].hist(r["temperature"][half], bins=25, density=True, alpha=0.5, label=name,
                     color=PALETTE[list(runs).index(name)])
axes[1].axvline(T_target, color=PALETTE[1], lw=2.5, label="velocity rescaling")
axes[1].set_xlabel(r"$T_\mathrm{inst}$ (second half)")
axes[1].set_ylabel("density")
fig.tight_layout()
save_figure(fig, "01_thermostat_comparison")
plt.show()

print(f"{'':20s} {'<T>':>7} {'std(T)/<T>':>11}")
for name, r in runs.items():
    T_ = r["temperature"][half]
    print(f"{name:20s} {T_.mean():7.3f} {T_.std() / T_.mean():11.3f}")
print(f"canonical expectation for the relative fluctuation: sqrt(2/f) = {np.sqrt(2 / dof):.3f}")

**Interpretation.**

- NVE: no temperature control; the microcanonical fluctuations of $T_\mathrm{inst}$ are smaller than canonical ones.
- Velocity rescaling: $T_\mathrm{inst}$ is pinned, and its fluctuation is artificially zero.
- Langevin: $T_\mathrm{inst}$ fluctuates around the target with a relative width comparable to the canonical
  $\sqrt{2/f}$ (a finite window of correlated samples gives only a rough estimate of a fluctuation).

Each choice also modifies the dynamics: rescaling rescales all velocities at once, and Langevin friction
damps momentum and perturbs transport properties. Which thermostat is appropriate depends on what is measured.

## 7. The thermostat also sets a timescale

Start from velocities at twice the target temperature and follow the relaxation for several friction coefficients.

In [ ]:
v_hot, _ = rescale_to_temperature(v_init, 1.0, 2.0 * T_target, dof)
fig, ax = plt.subplots(figsize=(6.5, 3))
for zeta in (0.2, 1.0, 5.0):
    r = lj.run_lj(x_init, v_hot, L, 1500, dt, thermostat="langevin", T_target=T_target,
                  friction=zeta, rng=np.random.default_rng(seed), sample_stride=5)
    ax.plot(r["time"], r["temperature"], lw=1.2, label=rf"$\zeta$ = {zeta}")
ax.axhline(T_target, color=REFERENCE, ls="--", lw=1)
ax.set_xlabel("time")
ax.set_ylabel(r"$T_\mathrm{inst}$")
ax.legend()
fig.tight_layout()
plt.show()

Stronger friction pulls the temperature to the target faster, but it also slows down diffusive motion of the particles.
Equilibrium averages of configurational quantities are the same for any $\zeta>0$; time-dependent properties are not.

## 8. A weak-coupling (Berendsen) barostat

After each step, positions and the box are scaled isotropically by

$$\mu = \left[1 - \frac{\kappa_T\,\Delta t}{\tau_P}\,(P_\mathrm{target} - P)\right]^{1/d}.$$

Here it is combined with the Langevin thermostat, and $P_\mathrm{target}$ is switched halfway through the run.

> The Berendsen weak-coupling barostat is used here for intuitive visualization and does not rigorously generate the exact NPT ensemble.

In [ ]:
n_half = barostat_steps // 2
P_schedule = np.r_[np.full(n_half, P_targets[0]), np.full(barostat_steps - n_half, P_targets[1])]
baro = lj.run_lj(x_init, v_init, L, barostat_steps, dt, thermostat="langevin", T_target=T_target,
                 friction=friction, rng=rng, barostat=True, P_target=P_schedule, tau_P=tau_P,
                 compressibility=compressibility, sample_stride=sample_stride)

tb = baro["time"]
t_switch = n_half * dt
window = 50  # samples
running = np.convolve(baro["pressure"], np.ones(window) / window, mode="valid")
fig, axes = plt.subplots(2, 1, figsize=(7, 5), sharex=True)
axes[0].plot(tb, baro["box_length"] ** 2, color=PALETTE[0])
axes[0].set_ylabel(r"box area $A = L^2$")
axes[1].plot(tb, baro["pressure"], color=MUTED, lw=0.6, label="instantaneous")
axes[1].plot(tb[window - 1:], running, color=PALETTE[0], label=f"running mean ({window * sample_stride * dt:.1f} time units)")
axes[1].plot(tb, np.where(tb <= t_switch, *P_targets), color=REFERENCE, ls="--", lw=1.2, label="target")
axes[1].set_ylabel(r"pressure $P$")
axes[1].set_xlabel("time")
axes[1].legend(ncol=3, fontsize=8)
fig.tight_layout()
save_figure(fig, "01_berendsen_barostat")
plt.show()

for label, sel in [(f"P_target = {P_targets[0]}", (tb > 0.5 * t_switch) & (tb <= t_switch)),
                   (f"P_target = {P_targets[1]}", tb > t_switch + 0.5 * (tb[-1] - t_switch))]:
    print(f"{label}: <P> = {baro['pressure'][sel].mean():.2f}, <A> = {np.mean(baro['box_length'][sel] ** 2):.1f}, "
          f"density = {N / np.mean(baro['box_length'][sel] ** 2):.3f}")

When the target pressure is raised, the box shrinks until the *average* pressure matches the target. The
instantaneous pressure of 36 particles fluctuates strongly; the barostat reacts to it with a lag set by $\tau_P$.
Weak coupling suppresses volume fluctuations relative to a true NPT ensemble, which is why production
simulations use other barostats (e.g. stochastic cell rescaling, Parrinello–Rahman or MTK).

## 9. Optional experiment

Try `tau_P = 0.5` or `tau_P = 10.0` in the parameter cell: a short coupling time follows the target quickly
but reacts to noise; a long one responds slowly. Try `friction = 0.1` or `5.0` and compare Section 6.

## Takeaways

- Periodic boundaries with the minimum-image convention require a box larger than twice the cutoff.
- NVE conserves energy (approximately, for a good timestep) but does not control temperature.
- Repeated exact velocity rescaling pins $T_\mathrm{inst}$ and removes canonical fluctuations; Langevin dynamics reproduces them approximately here.
- Thermostats and barostats change dynamics and timescales, not only averages.
- The Berendsen barostat gives an intuitive picture of pressure control but not the exact NPT ensemble.

*References:* Berendsen et al., *J. Chem. Phys.* **81**, 3684 (1984), doi:10.1063/1.448118 ·
Leimkuhler & Matthews, *Appl. Math. Res. Express* **2013**, 34, doi:10.1093/amrx/abs010 (BAOAB).